# Colab starter — Segmentación de heridas en ratas

Correr las celdas en orden. Este notebook clona el repo de GitHub, monta Google Drive (donde están `Images/` y `Masks/`), instala dependencias, y deja todo listo para llamar a los scripts de `src/`.

**Antes de correr:** en Colab, andá a `Entorno de ejecución -> Cambiar tipo de entorno de ejecución -> GPU` para tener GPU disponible.

In [1]:
# 1) Clonar el repo (reemplazar por tu URL real una vez que lo subas a GitHub)
REPO_URL = "https://github.com/camichahwan/wound_segmentation.git"
!git clone $REPO_URL
%cd wound_segmentation

Cloning into 'wound_segmentation'...
remote: Enumerating objects: 107, done.
remote: Counting objects: 100% (107/107), done.
remote: Compressing objects: 100% (62/62), done.
remote: Total 107 (delta 42), reused 98 (delta 33), pack-reused 0 (from 0)
Receiving objects: 100% (107/107), 4.40 MiB | 21.28 MiB/s, done.
Resolving deltas: 100% (42/42), done.
/content/wound_segmentation


In [2]:
# 2) Instalar dependencias
!pip install -q -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 6.7 MB/s eta 0:00:00


In [4]:
# 3) Montar Google Drive (te va a pedir autorización la primera vez)
import os, sys

REPO_DIR = "/content/wound_segmentation"
os.chdir(REPO_DIR)  # por si esta celda se corre sola, sin haber corrido la 1 justo antes
sys.path.insert(0, os.path.join(REPO_DIR, "src"))

from config import mount_drive_if_colab, get_data_root, get_images_dir, get_masks_dir, get_checkpoints_dir

mount_drive_if_colab()
print("Data root:", get_data_root())
print("Images:", get_images_dir())
print("Masks:", get_masks_dir())
print("Checkpoints (train.py guarda ACÁ, en Drive -- no en /content, que se borra si se desconecta la sesión):", get_checkpoints_dir())

Mounted at /content/drive
Data root: /content/drive/MyDrive/Tesis Imagenes
Images: /content/drive/MyDrive/Tesis Imagenes/Images
Masks: /content/drive/MyDrive/Tesis Imagenes/Masks
Checkpoints (train.py guarda ACÁ, en Drive -- no en /content, que se borra si se desconecta la sesión): /content/drive/MyDrive/Tesis Imagenes/model_checkpoints


In [5]:
# 4) Verificar que el emparejamiento imagen-máscara funciona antes de entrenar nada
from dataset import list_paired_and_unpaired, train_val_test_split

paired, unpaired = list_paired_and_unpaired()
print(f"Imágenes con máscara: {len(paired)}")
print(f"Imágenes sin máscara todavía: {len(unpaired)}")

train_samples, val_samples, test_samples = train_val_test_split(paired)
print(f"Train: {len(train_samples)} | Val: {len(val_samples)} | Test: {len(test_samples)}")

Imágenes con máscara: 739
Imágenes sin máscara todavía: 562
Train: 519 | Val: 110 | Test: 110


In [ ]:
# 5) Entrenar la U-Net preentrenada (rápido, buen primer experimento)
!cd src && python train.py --model pretrained --encoder resnet34 --epochs 40 --batch_size 8 --image_size 512

In [ ]:
# 6) Entrenar la U-Net desde cero (más lenta en converger, dejarla correr más épocas)
!cd src && python train.py --model scratch --epochs 80 --batch_size 8 --image_size 512

In [ ]:
# 7) Comparar ambos modelos en el set de test
# Los checkpoints ahora se guardan en Drive (ver celda 3, "Checkpoints"), no en /content,
# así que evaluate.py los busca en esa misma carpeta de Drive.
from config import get_checkpoints_dir

ckpt_dir = get_checkpoints_dir()
pretrained_ckpt = os.path.join(ckpt_dir, "unet_pretrained.pt")
scratch_ckpt = os.path.join(ckpt_dir, "unet_scratch.pt")

!cd src && python evaluate.py --checkpoints "{pretrained_ckpt}" "{scratch_ckpt}"

In [ ]:
# 8) Post-procesamiento + ensemble: ¿ayudan a los errores que NO son de heridas chicas?
# Reutiliza los dos checkpoints ya entrenados (no reentrena nada). Compara contra
# unet_pretrained solo con test pareado de Wilcoxon -- ver outputs/error_analysis/
# despues de correrlo para el reporte con la decision.
!cd src && python postprocess_and_ensemble.py

# Enfoque en dos etapas (heridas chicas/tempranas)

Ver `outputs/error_analysis/heridas_chicas_resize.md`: las heridas más chicas del dataset quedan en ~6x6 píxeles después del resize a 512x512 que usa el pipeline de una sola etapa -- por debajo de lo que la red puede ver. Estas celdas entrenan y evalúan un enfoque en dos etapas: una localización gruesa a mayor resolución, seguida de un recorte + segmentación fina sobre esa región. Correr en orden, cada celda depende de la anterior.

In [ ]:
# 9) Etapa 1: entrenar el localizador a mayor resolución (1024, no 512)
# batch_size mas chico que en la celda 5 porque 1024x1024 pesa mucho mas en memoria de GPU.
# Si tira error de memoria (OOM) en la T4, bajar --batch_size a 2.
!cd src && python train.py --model pretrained --encoder resnet34 --epochs 30 --batch_size 4 \
    --image_size 1024 --run_name unet_localizer_1024

In [ ]:
# 10) Generar el dataset de recortes (a partir de las máscaras reales) para entrenar la etapa 2
!cd src && python make_crop_dataset.py --crop_size 512

In [ ]:
# 11) Etapa 2: fine-tuning de unet_pretrained sobre los recortes (parte de los pesos ya entrenados,
# no de cero -- ver --init_checkpoint). WOUND_DATA_ROOT apunta train.py al dataset de recortes
# generado en la celda anterior, en vez de a las imágenes completas.
import os
from config import get_crops_data_root, get_checkpoints_dir, get_data_root

# OJO: hay que resolver ckpt_dir (y fijarlo con WOUND_CHECKPOINTS_DIR) ANTES
# de pisar WOUND_DATA_ROOT. get_checkpoints_dir() depende de get_data_root(),
# así que si no lo fijamos antes, train.py (que corre como subproceso y
# hereda estas variables de entorno) buscaría unet_pretrained.pt Y guardaría
# el checkpoint nuevo dentro de "Tesis Imagenes Crops/model_checkpoints" en
# vez de "Tesis Imagenes/model_checkpoints", que es donde están los demás
# checkpoints y donde después evaluate_two_stage.py los va a buscar.
ckpt_dir = get_checkpoints_dir()
pretrained_ckpt = os.path.join(ckpt_dir, "unet_pretrained.pt")

os.environ["WOUND_CHECKPOINTS_DIR"] = ckpt_dir
os.environ["WOUND_DATA_ROOT"] = get_crops_data_root()
print("Entrenando etapa 2 sobre:", os.environ["WOUND_DATA_ROOT"])
print("Checkpoints se guardan en:", os.environ["WOUND_CHECKPOINTS_DIR"])
print("Partiendo de checkpoint:", pretrained_ckpt)

!cd src && python train.py --model pretrained --encoder resnet34 --epochs 30 --batch_size 8 \
    --image_size 512 --run_name unet_stage2_finetuned --init_checkpoint "{pretrained_ckpt}"

# IMPORTANTE: sacar los dos overrides apenas termina, para que las celdas siguientes
# (evaluación de punta a punta) vuelvan a usar las imágenes completas originales
# y la carpeta de checkpoints normal, no la de los recortes.
del os.environ["WOUND_DATA_ROOT"]
del os.environ["WOUND_CHECKPOINTS_DIR"]
print("Overrides reseteados. Data root de nuevo:", get_data_root())
print("Checkpoints dir de nuevo:", get_checkpoints_dir())

In [ ]:
# 12) Evaluación de punta a punta: dos etapas vs. unet_pretrained solo, sobre las imágenes
# de test ORIGINALES completas (no recortes) -- esta es la comparación real.
ckpt_dir = get_checkpoints_dir()
stage1_ckpt = os.path.join(ckpt_dir, "unet_localizer_1024.pt")
stage2_ckpt = os.path.join(ckpt_dir, "unet_stage2_finetuned.pt")

!cd src && python evaluate_two_stage.py --stage1_checkpoint "{stage1_ckpt}" --stage2_checkpoint "{stage2_ckpt}"

In [ ]:
# 13) ¿El post-procesamiento arregla las fallas catastroficas nuevas del enfoque en dos etapas?
# Ver claude/analisis_resultados_dos_etapas.md en el proyecto: la celda 12 mejoro Dice/IoU
# de forma clara, pero introdujo 2 fallas catastroficas nuevas (DSC01378, DSC01037) por
# sobre-deteccion de la etapa 1. Esta celda prueba si el mismo post-procesamiento que ya
# funciono bien (celda 8) las arregla. No reentrena nada, es rapida (similar a la celda 8).
ckpt_dir = get_checkpoints_dir()
stage1_ckpt = os.path.join(ckpt_dir, "unet_localizer_1024.pt")
stage2_ckpt = os.path.join(ckpt_dir, "unet_stage2_finetuned.pt")

!cd src && python evaluate_two_stage_postproc.py --stage1_checkpoint "{stage1_ckpt}" --stage2_checkpoint "{stage2_ckpt}"

In [6]:
# 14) Diagnostico de la etapa 1: ¿por que fallan DSC01378 y DSC01037?
# No cambia ningun modelo: mide que tan grande predice la etapa 1 vs. la herida real y guarda
# imagenes superpuestas (verde=real, rojo=etapa 1, azul=recorte) para VER el problema antes de
# decidir el umbral de la validacion de tamaño. Rapida (solo inferencia).
ckpt_dir = get_checkpoints_dir()
stage1_ckpt = os.path.join(ckpt_dir, "unet_localizer_1024.pt")

!cd src && python diagnose_stage1.py --stage1_checkpoint "{stage1_ckpt}"

config.json: 100% 156/156 [00:00<00:00, 753kB/s]

model.safetensors: downloading bytes:  71% 61.5M/87.3M [00:01<00:00, 68.9MB/s, 3.30MB/s  ]
model.safetensors: downloading bytes: 100% 80.6M/80.6M [00:01<00:00, 51.2MB/s, 7.62MB/s  ]
model.safetensors: reconstructing file: 100% 87.3M/87.3M [00:01<00:00, 55.3MB/s, 8.38MB/s  ]
Diagnosticando etapa 1 sobre 110 imagenes de test.
  ... 20/110 imagenes procesadas
  ... 40/110 imagenes procesadas
  ... 60/110 imagenes procesadas
  ... 80/110 imagenes procesadas
  ... 100/110 imagenes procesadas

Detalle por imagen guardado en: /content/drive/MyDrive/Tesis Imagenes/model_checkpoints/stage1_diagnostics.csv

=== Distribucion: area predicha por etapa 1 / area real ===
count    110.000
mean       1.041
std        0.313
min        0.028
5%         0.791
25%        0.940
50%        1.003
75%        1.060
90%        1.137
95%        1.327
max        2.978
Name: area_ratio_pred_over_gt, dtype: float64

=== Distribucion: fraccion de la imagen que ocupa e

In [7]:
!cd /content/wound_segmentation && git pull

remote: Enumerating objects: 10, done.
remote: Counting objects: 100% (10/10), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 6 (delta 3), reused 6 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 3.51 KiB | 598.00 KiB/s, done.
From https://github.com/camichahwan/wound_segmentation
   753bff2..c497516  main       -> origin/main
Updating 753bff2..c497516
Fast-forward
 notebooks/colab_starter.ipynb      |  17 ++++
 src/evaluate_two_stage_s1filter.py | 162 +++++++++++++++++++++++++++++++++++++
 2 files changed, 179 insertions(+)
 create mode 100644 src/evaluate_two_stage_s1filter.py
